In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.14 The Photon Gas and Planck's Law

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.14",
    title="The Photon Gas and Planck's Law",
    blurb="This is where the volume's story began: classical physics assigning kT to every "
    "mode of the electromagnetic field, and the field having infinitely many. We "
    "exhibit the catastrophe honestly, then cure it with parts the volume already "
    "owns — an oscillator per mode, a Bose occupation at zero chemical potential, a "
    "mode count from the arsenal — and the resulting law weighs the Sun to four "
    "digits, splits into two different 'peaks' depending on how you plot it, and "
    "fits the afterglow of the Big Bang to parts in ten thousand.",
    difficulty="advanced",
    estimate="195–235 min",
)

## Notebook overview

Movement IV opens where quantum theory itself opened. In 1900 the classical theory of radiation
was failing in public: equipartition (Volume V's own theorem, correct within its hypotheses) assigns $k_BT$ to every mode of the electromagnetic field, the field has infinitely many modes,
and the predicted energy density of a warm cavity diverges. This **ultraviolet catastrophe** is
the failure the whole volume has been walking toward, and this notebook finally stands at the
spot. In the manner [§7.13](semiconductor-statistics.ipynb) made standard, the point is that every tool needed to fix it is now
second-hand: each electromagnetic mode *is* the harmonic oscillator of [§7.5](oscillator-at-temperature.ipynb); the $\omega^2$ mode
counting is the density of states of [§7.3](statistical-toolkit.ipynb); the occupation is the Bose function of [§7.7](bose-einstein-fermi-dirac.ipynb) with $\mu = 0$, and the $\mu = 0$ is not an assumption but the forward flag of [§7.7](bose-einstein-fermi-dirac.ipynb), argued in full here (photon number
is unconserved, so $F$ minimizes over $N$ itself); and the integral that turns Planck's law
into the $T^4$ law is the $\pi^4/15$ of [§7.3](statistical-toolkit.ipynb), computed there before the physics that needed it
existed. Planck's law costs this volume one line of assembly, because three earlier notebooks
each supplied a third.

What the assembled law delivers is out of proportion to that one line. **Stefan–Boltzmann**
gives $\sigma = 5.670374\times10^{-8}\ \text{W/m}^2\text{K}^4$ from constants alone (exact, in
fact, in the 2019 SI), and with it the Sun submits to two numbers and a law:
$L = 3.8280\times10^{26}$ W against the measured $3.828\times10^{26}$, and the solar constant
$1361\ \text{W/m}^2$ at one astronomical unit. **Wien displacement** arrives with its classic
gotcha taught in full: maximizing $u_\lambda$ and maximizing $u_\nu$ give *different roots* ($x^* = 4.9651$ versus $2.8214$), so the wavelength peak and the frequency peak are different photons (for the Sun, 502 nm versus 880 nm), because a spectral density's maximum depends on
the variable it is a density *in*. Photon thermodynamics ties the movement backward
($P = u/3$ is the ultrarelativistic relation of [§7.11](white-dwarfs-chandrasekhar.ipynb) at $\mu = 0$: the kinematics that doomed the
white dwarfs, now holding starlight) and forward: the adiabat $TV^{1/3} = \text{const}$ is why
cosmic expansion cools the blackbody while preserving its Planck shape exactly. Which is why
the data jewel exists at all: the **cosmic microwave background**, 3000 K light from
recombination stretched a thousandfold into a 2.725 K spectrum, measured by COBE/FIRAS as the
most perfect blackbody ever recorded. The actual FIRAS monopole table (Fixsen et al. 1996) is
embedded and fit with `scipy.optimize.curve_fit` over the single parameter $T$ — after the fit
machinery is certified on the canonical peak and on synthetic data — recovering $T = 2.725$ K
with residuals at parts in $10^4$ of the peak. Along the way a numerical trap is taught with
relish, because verification actually hit it: `quad` on the *dimensional* Planck integrand
returns an answer **thirty-seven orders of magnitude wrong, with no warning**; scaled to
order one it is exact to ten digits. The stretch balances a planet, and the 33 K it cannot
explain has a name.

> **Conventions (this notebook).** SI units throughout; CODATA/IAU constants in
> Setup ($h$, $c$, $k_B$ exact by definition since 2019). The spectral bookkeeping is spelled
> out because it is this notebook's own gotcha: $u(\omega)$, $u(\nu)$, and $u(\lambda)$ are
> *densities in different variables*, related by Jacobians, and their peaks differ. Every
> Planck denominator uses `numpy.expm1` (the standing rule of [§7.5](oscillator-at-temperature.ipynb)); `scipy.integrate.quad` is
> applied **only to nondimensionalized integrands** (the trap demonstrated once, then the
> rule); both Wien roots come from `scipy.optimize.brentq` with their transcendental equations
> stated; the FIRAS fit uses `scipy.optimize.curve_fit` with `p0` stated and the covariance
> read. Unit conversions happen at single stated points: cm⁻¹ → Hz by one factor of $100c$,
> and W m⁻² Hz⁻¹ sr⁻¹ → MJy/sr inside `B_nu_MJy` alone (1 Jy = $10^{-26}$ W m⁻² Hz⁻¹).
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the Rayleigh–Jeans and Wien limits against Planck at stated $\hbar\omega/
> k_BT$; $\sigma$ against its SI-exact value to eight digits; the Sun against its measured
> luminosity; both Wien constants against CODATA; the photon count against the canonical
> 411 cm⁻³; the Planck-in-MJy/sr function against the canonical CMB peak *before* any fitting;
> the fit procedure against synthetic data of known temperature; and the dimensional-quad
> failure gated as being wrong by more than thirty orders. A ✓ is strong evidence; a ✗ is a
> prompt to *locate the discrepancy*.
>
> **Scope.** The greenhouse gap is named and pointed outward (climate physics cited, not
> developed); CMB anisotropies, stellar atmospheres, and opacity are named horizons. See
> Planck (1900, 1901); Fixsen et al. 1996, ApJ 473, 576 (the FIRAS monopole, Table 4); Pathria
> & Beale (Ch. 7); Kardar (Ch. 7). Cross-reference [§7.5](oscillator-at-temperature.ipynb) (the oscillator per mode; `expm1`),
> [§7.7](bose-einstein-fermi-dirac.ipynb) ($\mu = 0$, flagged there, argued here), [§7.3](statistical-toolkit.ipynb) (three invocations: the mode count,
> $\pi^4/15$, $\zeta(3)$), [§7.11](white-dwarfs-chandrasekhar.ipynb) ($P = u/3$), [§5.5](../05-classical-stat-mech/ergodicity.ipynb) (equipartition, the honest villain), [§7.13](semiconductor-statistics.ipynb)
> (unit discipline, inherited), and forward to [§7.15](einstein-a-b-coefficients.ipynb) (Einstein's A and B), [§7.16](phonons-debye.ipynb) (phonons),
> [§7.17](bose-einstein-condensation.ipynb) (BEC).

## Theory in brief

### The catastrophe, exhibited

A cavity's electromagnetic field decomposes into normal modes; the counting of [§7.3](statistical-toolkit.ipynb) (re-derived
below in two lines, two polarizations included) gives the mode density $g(\omega) =
V\omega^2/\pi^2c^3$. Volume V's equipartition ([§5.5](../05-classical-stat-mech/ergodicity.ipynb)) assigns each mode $k_BT$, so

```{math}
:label: eq-rayleigh-jeans
u_{\text{RJ}}(\omega, T) = \frac{\omega^2}{\pi^2c^3}\,k_BT
\qquad\Longrightarrow\qquad
\int_0^\infty u_{\text{RJ}}\,d\omega = \infty .
```

This is the **Rayleigh–Jeans law**, and it is not a wrong guess: it is the *correct low-frequency limit* of the true law (Planck/RJ = 0.995 at $\hbar\omega/k_BT = 0.01$, verified
below). Its integral is the catastrophe: every cavity should contain infinite energy, mostly
at ultraviolet and shorter wavelengths. This was Kelvin's second "cloud" over 1900 physics,
and it was not a technicality: equipartition is a theorem, the mode count is geometry, and together they are wrong about every glowing object in the universe. Wien had guessed an
empirical exponential that captured the high-frequency tail; what nobody had was a reason.

### μ = 0, argued in full

Photon number is not conserved: cavity walls absorb and emit freely, so $N$ is not a
constraint but an internal variable. Equilibrium minimizes $F(T, V, N)$ over $N$ itself,

```{math}
:label: eq-mu-zero
\left(\frac{\partial F}{\partial N}\right)_{T,V} = 0
\qquad\Longleftrightarrow\qquad
\mu = 0 ,
```

which is exactly the definition of the chemical potential (the lineage of [§7.4](thermal-density-matrix.ipynb)) evaluated at the
free minimum. This is the argument [§7.7](bose-einstein-fermi-dirac.ipynb) flagged forward and this notebook supplies in full. It
also kills something: with no protected number there is no filling constraint to force
condensation, so a black cavity has no photon BEC; the ceiling physics of [§7.17](bose-einstein-condensation.ipynb) requires a *conserved* boson number. (Dye-filled microcavities engineer an effectively conserved photon
number and do condense: one breath, outward.)

### Planck's law from second-hand parts

Each mode of frequency $\omega$ is a harmonic oscillator, literally: the field's normal-mode coordinates obey the Hamiltonian of [§7.5](oscillator-at-temperature.ipynb). Its thermal occupation is the Bose function of [§7.7](bose-einstein-fermi-dirac.ipynb) at
$\mu = 0$, $n_B = 1/(e^{\hbar\omega/k_BT} - 1)$. Multiply occupation, energy per photon, and
the mode density of [§7.3](statistical-toolkit.ipynb):

```{math}
:label: eq-planck-law
u(\omega, T) = \frac{\hbar\omega^3}{\pi^2c^3}\,
\frac{1}{e^{\hbar\omega/k_BT} - 1} ,
```

**Planck's law** — with `numpy.expm1` in the denominator, the standing rule of [§7.5](oscillator-at-temperature.ipynb) against
catastrophic cancellation at $\hbar\omega \ll k_BT$. The limits close the history: for
$x = \hbar\omega/k_BT \to 0$ the occupation is $k_BT/\hbar\omega$ and Rayleigh–Jeans emerges
(ratios 0.995, 0.951, 0.582, 0.034 at $x = 0.01, 0.1, 1, 5$, verified); for $x \gg 1$ Wien's
exponential appears with its reason attached. Planck found this expression in October 1900 by
interpolating between the two limits, and spent the next months deriving it by an assumption
he later called "an act of desperation": energy exchanged in quanta $\hbar\omega$. The volume
assembles the same law in one line because [§7.5](oscillator-at-temperature.ipynb), [§7.7](bose-einstein-fermi-dirac.ipynb), and [§7.3](statistical-toolkit.ipynb) each supplied a part.

### Stefan–Boltzmann, and the Sun weighed

Integrate {eq}`eq-planck-law` with the substitution $x = \hbar\omega/k_BT$:

```{math}
:label: eq-stefan-boltzmann
u = aT^4,\quad a = \frac{\pi^2k_B^4}{15\hbar^3c^3},
\qquad
\sigma = \frac{ac}{4} = 5.670374\times10^{-8}\ \text{W m}^{-2}\text{K}^{-4},
```

where the $\int_0^\infty x^3\,dx/(e^x - 1) = \pi^4/15$ is the Bose integral of [§7.3](statistical-toolkit.ipynb), invoked rather
than re-derived. In the 2019 SI, $h$, $c$, and $k_B$ are *defined* constants, so $\sigma$ is now exact, a combination of definitions rather than a measured quantity (verified to eight
digits below). The Sun then submits to the formula: with the IAU $T_{\text{eff}} = 5772$ K and
$R_\odot = 6.957\times10^8$ m, $L = 4\pi R_\odot^2\sigma T^4 = 3.8280\times10^{26}$ W against
the measured $3.828\times10^{26}$, and the flux at one astronomical unit is the solar constant,
$1361\ \text{W/m}^2$. The honest note: $T_{\text{eff}}$ *means* "the temperature a blackbody
of the Sun's size and luminosity would have"; the real photosphere is layered, and the match
is partly definitional; the non-trivial content is that one temperature describes the
spectrum's shape as well as its total.

### The taught trap: quad meets Planck

The trap is sprung deliberately: we hand `scipy.integrate.quad` the dimensional Planck
integrand {eq}`eq-planck-law` over $(0, \infty)$, exactly as a first instinct would, and
compare its report against the Stefan–Boltzmann value the integral provably equals:

```{math}
:label: eq-quad-trap
\texttt{quad}(u_{\text{Planck}},\ 0,\ \infty) = 8\times10^{-38}
\qquad\text{versus}\qquad
aT^4 = 0.84\ \text{J/m}^3
\quad (T = 5772\ \text{K}).
```

Thirty-seven orders of magnitude wrong, and *silently*: no warning is raised (verified
below). The adaptive sampler probes the transformed infinite interval at points that all miss
the integrand's support near $\omega \sim 2\times10^{15}$ rad/s, concludes the function is
essentially zero, and reports success. Nondimensionalize ($x = \hbar\omega/k_BT$) and the same
machine returns $\pi^4/15$ to ten digits. Stated as a standing rule of the course: **quad
meets physics only after the physics has been scaled to order one** — a wrong answer with no
warning is the most dangerous kind.

### Wien displacement: which peak?

Maximize the *wavelength* density $u_\lambda \propto x^5/(e^x - 1)$ and the *frequency*
density $u_\nu \propto x^3/(e^x - 1)$ (both with $x = hc/\lambda k_BT = h\nu/k_BT$):

```{math}
:label: eq-wien
5\,(1 - e^{-x}) = x \;\Rightarrow\; x^*_\lambda = 4.9651,
\qquad
3\,(1 - e^{-x}) = x \;\Rightarrow\; x^*_\nu = 2.8214 ,
```

giving $\lambda_{\max}T = b = 2.8978$ mm·K (CODATA: 2.897772) and $\nu_{\max}/T = 58.79$
GHz/K. The gotcha, taught in full: these are **different photons** (for the Sun, $\lambda_{\max} = 502$ nm while $c/\nu_{\max} \approx 880$ nm), because a density's maximum
depends on the variable it is a density *in*: $u_\lambda = u_\nu\,|d\nu/d\lambda|$, and the
Jacobian $c/\lambda^2$ reshapes the curve. Neither peak is "the" peak; each answers a
different question (energy per unit wavelength versus per unit frequency). This Jacobian
moral will matter in every spectroscopy the reader ever does. The evolutionary aside, one
sentence: 502 nm is blue-green, the band into which daylight vision evolved.

### Photon thermodynamics: pressure, entropy, expansion

The assembled gas is a complete thermodynamic system, and standard machinery reads off its
state functions: integrating the occupation over the mode count gives the photon number,
while kinetic theory and the free energy deliver pressure and entropy (Pathria & Beale,
Ch. 7, runs the full derivations):

```{math}
:label: eq-photon-thermo
\frac{N}{V} = \frac{2\zeta(3)}{\pi^2}\left(\frac{k_BT}{\hbar c}\right)^3,
\qquad
P = \frac{u}{3},
\qquad
S = \frac{4}{3}aT^3V
\;\Rightarrow\;
TV^{1/3} = \text{const (adiabat)}.
```

The number density is the $\zeta(3)$ integral of [§7.3](statistical-toolkit.ipynb). The pressure is the *ultrarelativistic* relation of [§7.11](white-dwarfs-chandrasekhar.ipynb) (momentum flux for particles with $\varepsilon = pc$ gives $P = u/3$ regardless of statistics), now at $\mu = 0$: the same $4/3$-polytrope kinematics that doomed the white
dwarfs holds starlight. The adiabat explains the sky: in an expanding universe every mode's
wavelength stretches as the scale factor $a(t)$ while the occupation of each comoving mode is
preserved, so a Planck spectrum maps onto a Planck spectrum with $T \propto 1/a$: expansion cools the blackbody *without deforming it* (demonstrated numerically below, and the reason
the next item exists).

### The jewel: FIRAS

A spectrometer pointed at the sky measures not the energy density $u$ but the specific
intensity, energy per unit frequency and solid angle; converting {eq}`eq-planck-law` to
those variables (a Jacobian plus a factor $c/4\pi$) gives the brightness against which the
cosmos is tested:

```{math}
:label: eq-firas-fit
B_\nu(T) = \frac{2h\nu^3}{c^2}\,\frac{1}{e^{h\nu/k_BT} - 1}
\quad\longrightarrow\quad
T_{\text{CMB}} = 2.725\ \text{K},
```

fit to the actual COBE/FIRAS monopole spectrum: 3000 K light from recombination, stretched by
$\sim$1100 into the most perfect blackbody ever recorded. The notebook embeds the monopole
table (Fixsen et al. 1996, Table 4; 43 points, 2.27–21.33 cm⁻¹, MJy/sr) and fits the single
parameter $T$ with `scipy.optimize.curve_fit`, the procedure certified first against the canonical peak (160.2 GHz, 383.9 MJy/sr) and on synthetic data of known temperature. The
cosmic bookkeeping read out: $n_\gamma = 411$ photons/cm³, everywhere, including the room the
reader sits in; and against the baryon density this is $\sim1.6\times10^9$ photons per baryon
— cosmology's great dimensionless number. The universe is, by count, almost entirely thermal
light.

### The stretch: a planet in the balance

A planet in steady state radiates exactly what it absorbs: it intercepts sunlight over its
disc $\pi R_E^2$, reflects the fraction $A$ (the albedo), and reradiates thermally over its
entire surface $4\pi R_E^2$. Setting absorption against Stefan–Boltzmann emission,

```{math}
:label: eq-earth-balance
\pi R_E^2\,(1 - A)\,S_0 = 4\pi R_E^2\,\sigma T_{\text{eq}}^4
\qquad\Longrightarrow\qquad
T_{\text{eq}} = T_\odot\sqrt{\frac{R_\odot}{2d}}\,(1 - A)^{1/4},
```

which evaluates to 278.3 K for a black Earth and 254.6 K at the observed albedo $A = 0.3$.
The observed mean surface temperature is 288 K, and the 33 K difference *is* the greenhouse
effect: the atmosphere passes sunlight in the visible and absorbs the planet's own infrared —
a frequency-selective blanket, named here and pointed outward.

## Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq, curve_fit
from scipy.special import zeta

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Constants. h, c, k_B are EXACT by definition in the 2019 SI; the
# solar values are IAU nominal; the astronomical unit is exact by IAU resolution.
from scipy.constants import h as H  # Planck constant, J·s (exact)
from scipy.constants import c as C  # speed of light, m/s (exact)
from scipy.constants import k as KB  # Boltzmann constant, J/K (exact)

HBAR = H / (2.0 * np.pi)  # derived from the exact h (the consistency rule of §7.13)
R_SUN = 6.957e8  # nominal solar radius, m (IAU 2015)
T_SUN = 5772.0  # nominal solar effective temperature, K (IAU 2015)
AU = 1.495978707e11  # astronomical unit, m (exact, IAU 2012)
T_CMB_REF = 2.72548  # CMB temperature, K (Fixsen 2009) — used only for synthetic tests
N_BARYON = 2.47e-7  # cosmic baryon density, cm^-3 (Planck 2018, Ω_b h^2 = 0.0224)
JY = 1.0e-26  # 1 jansky in W m^-2 Hz^-1 — the MJy/sr conversion lives in B_nu_MJy alone
CM1_TO_HZ = 100.0 * C  # 1 cm^-1 in Hz — the single wavenumber conversion point


def u_planck(omega, T):
    """Planck's spectral energy density u(ω, T) = (ħω^3/π^2c^3)/(e^{ħω/k_BT} − 1), SI.

    eq-planck-law, assembled from the oscillator occupation of §7.5, the Bose function of §7.7
    at μ = 0, and the mode density of §7.3, Vω^2/π^2c^3. The denominator uses numpy.expm1
    (the standing rule of §7.5): at ħω << k_BT the naive e^x − 1 cancels catastrophically
    and would corrupt exactly the Rayleigh–Jeans limit this law must reproduce.

    Parameters
    ----------
    omega : float or numpy.ndarray
        Angular frequency, rad/s.
    T : float
        Temperature, K.

    Returns
    -------
    float or numpy.ndarray
        Spectral energy density per unit angular frequency, J·s/m^3.
    """
    x = HBAR * np.asarray(omega, dtype=float) / (KB * T)
    return HBAR * np.asarray(omega, dtype=float) ** 3 / (np.pi**2 * C**3) / np.expm1(x)


def u_RJ(omega, T):
    """The Rayleigh–Jeans density u_RJ(ω, T) = ω^2 k_BT/π^2c^3 (eq-rayleigh-jeans).

    Volume V's equipartition (k_BT per mode, §5.5) on the mode count of §7.3: the correct
    low-frequency limit of Planck's law, and the ultraviolet catastrophe when
    integrated — its ω^2 growth never turns over.

    Parameters
    ----------
    omega : float or numpy.ndarray
        Angular frequency, rad/s.
    T : float
        Temperature, K.

    Returns
    -------
    float or numpy.ndarray
        Spectral energy density per unit angular frequency, J·s/m^3.
    """
    return np.asarray(omega, dtype=float) ** 2 * KB * T / (np.pi**2 * C**3)


def u_wien(omega, T):
    """Wien's empirical exponential u_W(ω, T) = (ħω^3/π^2c^3) e^{−ħω/k_BT}.

    The 1896 guess that captured the high-frequency tail while awaiting a reason:
    Planck's law with the −1 dropped from the denominator, i.e. the occupation's
    Boltzmann limit. Correct for ħω >> k_BT, low by a factor x at small x.

    Parameters
    ----------
    omega : float or numpy.ndarray
        Angular frequency, rad/s.
    T : float
        Temperature, K.

    Returns
    -------
    float or numpy.ndarray
        Spectral energy density per unit angular frequency, J·s/m^3.
    """
    w = np.asarray(omega, dtype=float)
    return HBAR * w**3 / (np.pi**2 * C**3) * np.exp(-HBAR * w / (KB * T))


def B_nu_MJy(nu_Hz, T):
    """The Planck brightness B_ν(T) = (2hν^3/c^2)/(e^{hν/k_BT} − 1), in MJy/sr.

    The FIRAS-unit Planck function: brightness per unit frequency per steradian,
    converted from W m^-2 Hz^-1 sr^-1 to MJy/sr at THIS single point (1 Jy =
    1e-26 W m^-2 Hz^-1, so the factor is 1/(1e-26 · 1e6) = 1e20). Certified in
    Exercise 6 against the canonical CMB peak (160.2 GHz, 383.9 MJy/sr) BEFORE any
    fitting — a wrong power of ten is invisible on a log plot, and the check costs
    one line. numpy.expm1 in the denominator, as everywhere.

    Parameters
    ----------
    nu_Hz : float or numpy.ndarray
        Frequency in Hz (convert cm^-1 inputs via CM1_TO_HZ first — the single
        stated wavenumber conversion point).
    T : float
        Temperature, K.

    Returns
    -------
    float or numpy.ndarray
        Brightness in MJy/sr.
    """
    nu = np.asarray(nu_Hz, dtype=float)
    x = H * nu / (KB * T)
    B_SI = 2.0 * H * nu**3 / C**2 / np.expm1(x)
    return B_SI / (JY * 1.0e6)


def stefan_sigma():
    """The Stefan–Boltzmann constant σ = 2π^5 k_B^4 / (15 h^3 c^2), W m^-2 K^-4.

    Assembled from the SI-2019 defined constants, so the result is itself exact —
    a combination of definitions (eq-stefan-boltzmann). The π^4/15 inside is the Bose integral of §7.3
    invoked; Exercise 3 re-verifies it by nondimensionalized quad.

    Returns
    -------
    float
        σ in W m^-2 K^-4.
    """
    return 2.0 * np.pi**5 * KB**4 / (15.0 * H**3 * C**2)


def wien_roots():
    """Both Wien displacement roots by scipy.optimize.brentq (eq-wien).

    Maximizing u_λ ∝ x^5/(e^x − 1) gives 5(1 − e^{−x}) = x, bracketed on [4, 6];
    maximizing u_ν ∝ x^3/(e^x − 1) gives 3(1 − e^{−x}) = x, bracketed on [2, 4].
    Two different roots because the two spectra are densities in different
    variables — the notebook's own gotcha, quantified.

    Returns
    -------
    tuple
        (x_lambda, x_nu) = (4.9651..., 2.8214...).
    """
    x_lam = brentq(lambda x: 5.0 * (1.0 - np.exp(-x)) - x, 4.0, 6.0, xtol=1e-13)
    x_nu = brentq(lambda x: 3.0 * (1.0 - np.exp(-x)) - x, 2.0, 4.0, xtol=1e-13)
    return x_lam, x_nu


def photon_density(T):
    """The photon number density n = (2ζ(3)/π^2)(k_BT/ħc)^3, per m^3.

    the ζ(3) integral of §7.3 (∫x^2/(e^x − 1) = 2ζ(3)), invoked via scipy.special.zeta.
    At the CMB temperature this is the famous 411 photons per cubic centimetre —
    everywhere, including the room the reader sits in.

    Parameters
    ----------
    T : float
        Temperature, K.

    Returns
    -------
    float
        Photon number density, m^-3.
    """
    return (2.0 * zeta(3.0) / np.pi**2) * (KB * T / (HBAR * C)) ** 3


def earth_equilibrium(albedo):
    """A planet's radiative-equilibrium temperature T_eq = T_☉ √(R_☉/2d) (1 − A)^{1/4}.

    eq-earth-balance: absorbed disk power πR_E^2(1 − A)S_0 equals radiated sphere
    power 4πR_E^2 σT^4; the planet's radius cancels, and with S_0 = σT_☉^4 R_☉^2/d^2
    the result depends only on the Sun's surface temperature, the distance ratio,
    and the albedo.

    Parameters
    ----------
    albedo : float
        Bond albedo A (Earth: ~0.3).

    Returns
    -------
    float
        Equilibrium temperature, K.
    """
    return T_SUN * np.sqrt(R_SUN / (2.0 * AU)) * (1.0 - albedo) ** 0.25


# The COBE/FIRAS CMB monopole spectrum — Fixsen et al. 1996, ApJ 473, 576, Table 4,
# as distributed by NASA LAMBDA (firas_monopole_spec_v1.txt). Columns: frequency
# (cm^-1), monopole spectrum (MJy/sr), residual from the best-fit blackbody
# (kJy/sr), 1σ uncertainty (kJy/sr). 43 points, 2.27–21.33 cm^-1.
FIRAS = np.array(
    [
        [2.27, 200.723, 5, 14],
        [2.72, 249.508, 9, 19],
        [3.18, 293.024, 15, 25],
        [3.63, 327.770, 4, 23],
        [4.08, 354.081, 19, 22],
        [4.54, 372.079, -30, 21],
        [4.99, 381.493, -30, 18],
        [5.45, 383.478, -10, 18],
        [5.90, 378.901, 32, 16],
        [6.35, 368.833, 4, 14],
        [6.81, 354.063, -2, 13],
        [7.26, 336.278, 13, 12],
        [7.71, 316.076, -22, 11],
        [8.17, 293.924, 8, 10],
        [8.62, 271.432, 8, 11],
        [9.08, 248.239, -21, 12],
        [9.53, 225.940, 9, 14],
        [9.98, 204.327, 12, 16],
        [10.44, 183.262, 11, 18],
        [10.89, 163.830, -29, 22],
        [11.34, 145.750, -46, 22],
        [11.80, 128.835, 58, 23],
        [12.25, 113.568, 6, 23],
        [12.71, 99.451, -6, 23],
        [13.16, 87.036, 6, 22],
        [13.61, 75.876, -17, 21],
        [14.07, 65.766, 6, 20],
        [14.52, 57.008, 26, 19],
        [14.97, 49.223, -12, 19],
        [15.43, 42.267, -19, 19],
        [15.88, 36.352, 8, 21],
        [16.34, 31.062, 7, 23],
        [16.79, 26.580, 14, 26],
        [17.24, 22.644, -33, 28],
        [17.70, 19.255, 6, 30],
        [18.15, 16.391, 26, 32],
        [18.61, 13.811, -26, 33],
        [19.06, 11.716, -6, 35],
        [19.51, 9.921, 8, 41],
        [19.97, 8.364, 26, 55],
        [20.42, 7.087, 57, 88],
        [20.87, 5.801, -116, 155],
        [21.33, 4.523, -432, 282],
    ]
)

sigma_SB = stefan_sigma()
print(f"σ = {sigma_SB:.9e} W m^-2 K^-4  (exact in the 2019 SI)")
print(
    f"FIRAS monopole table loaded: {FIRAS.shape[0]} points, {FIRAS[0, 0]}–{FIRAS[-1, 0]} cm^-1"
)

## Exercise 1 — The catastrophe, exhibited

Equipartition meets an infinity of modes: the failure that started everything, derived
honestly. Cite {eq}`eq-rayleigh-jeans`.

1. Derive $u_{\text{RJ}} = \omega^2k_BT/\pi^2c^3$ from Volume V equipartition ($k_BT$ per
   mode, [§5.5](../05-classical-stat-mech/ergodicity.ipynb) invoked) on the mode count of [§7.3](statistical-toolkit.ipynb) (re-derived in two lines, polarizations counted).
2. Show the integral diverges (partial integrals to cutoff $\Lambda$ growing as $\Lambda^3$, exactly $\times8$ per doubling) and verify Planck/RJ = 0.995 at $\hbar\omega/k_BT = 0.01$:
   the classical law is the *correct limit*, not a wrong guess.
3. Plot the founding figure: Planck, Rayleigh–Jeans, and Wien's exponential at the solar
   $T = 5772$ K, catastrophe region shaded.
4. Tell the history straight (prose): 1900, the crisis, Planck's "act of desperation," and the volume's through-line: this notebook stands where "classical statistical mechanics
   failed" began.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    ratio_planck_rj,
    0.995,
    "Rayleigh–Jeans is Planck's low-frequency limit — and its integral is the catastrophe",
    rtol=1e-3,
)
validate.close(
    list(doubling_ratios),
    [8.0, 8.0],
    "the divergence quantified: the classical energy grows as Λ^3, ×8 per doubling, without bound",
    rtol=1e-9,
)

## Exercise 2 — μ = 0 argued in full, and Planck assembled from second-hand parts

The volume's economy on display: an oscillator, an occupation, a mode count. Cite
{eq}`eq-mu-zero`, {eq}`eq-planck-law`.

1. Make the $\mu = 0$ argument in full: unconserved photon number $\Rightarrow$ $F$ minimized
   over $N$ $\Rightarrow$ $\mu = 0$ (the argument [§7.7](bose-einstein-fermi-dirac.ipynb) flagged forward, supplied here); note
   the consequence (no protected number, no cavity BEC; the dye-microcavity exception in one
   outward breath).
2. Assemble $u(\omega,T) = (\hbar\omega^3/\pi^2c^3)/(e^{\hbar\omega/k_BT} - 1)$ from the oscillator occupation of [§7.5](oscillator-at-temperature.ipynb)
   and the $g(\omega)$ of [§7.3](statistical-toolkit.ipynb), with `numpy.expm1` in the denominator.
3. Verify the two limits across $\hbar\omega/k_BT = 0.01$–$5$ (ratios against Rayleigh–Jeans
   below; against Wien's exponential above).
4. State the assembly's meaning (prose): Planck's law costs this volume one line because
   three earlier notebooks each supplied a third.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    list(planck_over_rj),
    [0.995, 0.951, 0.582, 0.034],
    "Planck's law assembled, with the Rayleigh–Jeans limit verified across two decades",
    rtol=2e-2,
)
validate.close(
    planck_over_wien[-1],
    1.0068,
    "and the Wien limit verified above: the 1896 exponential, with its reason attached",
    rtol=1e-3,
)

## Exercise 3 — Stefan–Boltzmann, and a trap worth thirty-seven orders of magnitude

The $T^4$ law from the integral of [§7.3](statistical-toolkit.ipynb), the Sun weighed, and `quad` taught a lesson. Cite
{eq}`eq-stefan-boltzmann`, {eq}`eq-quad-trap`.

1. Demonstrate the trap: `quad` on the *dimensional* Planck integrand over $(0, \infty)$ at
   $T = 5772$ K returns $\sim8\times10^{-38}$ — silently wrong by thirty-seven orders (the
   sampler never finds the peak at $\omega \sim 2\times10^{15}$); capture any warnings and
   show there are none; state the standing rule (nondimensionalize first).
2. Integrate correctly in $x = \hbar\omega/k_BT$: recover $\int x^3/(e^x - 1) = \pi^4/15$ to
   ten digits ([§7.3](statistical-toolkit.ipynb) invoked) and $I/aT^4 = 1.0000000000$.
3. Compute $\sigma = 2\pi^5k_B^4/15h^3c^2 = 5.670374\times10^{-8}$ W m⁻²K⁻⁴ from constants
   (SI-2019 exactness noted).
4. Weigh the Sun: $L = 4\pi R_\odot^2\sigma T^4$ vs the measured $3.828\times10^{26}$ W, and
   the solar constant at 1 AU — with the $T_{\text{eff}}$ honesty note.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    orders_wrong > 30.0 and len(caught) == 0,
    "the trap is real: dimensional quad silently loses the Planck peak — scale to order one first",
    f"wrong by {orders_wrong:.0f} orders, {len(caught)} warnings",
)
validate.close(
    I_x,
    np.pi**4 / 15.0,
    "the Bose integral of §7.3, re-verified where it is finally spent: π^4/15 to ten digits",
    rtol=1e-10,
)
validate.close(
    ratio_nondim,
    1.0,
    "the nondimensionalized route recovers aT^4 exactly",
    rtol=1e-10,
)
validate.close(
    [sigma_SB, L_sun, S_0],
    [5.670374419e-8, 3.828e26, 1361.0],
    "σ from constants to eight digits; the Sun weighed by a law; the solar constant at 1 AU",
    rtol=1e-3,
)

## Exercise 4 — Wien displacement: which peak?

Two maximizations, two roots, two different photons: the gotcha every spectroscopist must
survive once. Cite {eq}`eq-wien`.

1. Maximize $u_\lambda$: derive $5(1 - e^{-x}) = x$, solve with `scipy.optimize.brentq` on
   $[4, 6]$, and form $b = hc/k_Bx^* = 2.8978$ mm·K vs CODATA 2.897772.
2. Maximize $u_\nu$: derive $3(1 - e^{-x}) = x$ (`brentq` on $[2, 4]$), and form
   $\nu_{\max}/T = 58.79$ GHz/K.
3. Exhibit the gotcha: for the Sun, $\lambda_{\max} = 502$ nm while $c/\nu_{\max} \approx
   880$ nm; plot $u_\lambda$ and $u_\nu$ side by side with both peaks marked, and state the
   Jacobian moral (a density's peak lives in its variable).
4. One sentence each (prose): green-adapted eyes; and the CMB's $\nu_{\max} = 160.2$ GHz as
   the bridge to Exercise 6.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [x_lam, x_nu, b_wien * 1e3],
    [4.9651, 2.8214, 2.8978],
    "Wien twice: the peak depends on the plotting variable",
    rtol=1e-3,
)
validate.close(
    b_wien,
    2.897771955e-3,
    "and the wavelength constant lands on CODATA to the displayed digits",
    rtol=1e-5,
)

## Exercise 5 — Photon thermodynamics: pressure, entropy, and an expanding sky

The gas's mechanics, with the relation of [§7.11](white-dwarfs-chandrasekhar.ipynb) reappearing and the CMB's survival explained. Cite
{eq}`eq-photon-thermo`.

1. Compute $N/V = (2\zeta(3)/\pi^2)(k_BT/\hbar c)^3$ (the $\zeta(3)$ of [§7.3](statistical-toolkit.ipynb) invoked via
   `scipy.special.zeta`) at 300 K (the room's photon count) and 2.725 K (the canonical
   411 cm⁻³).
2. Derive $P = u/3$ as the ultrarelativistic relation of [§7.11](white-dwarfs-chandrasekhar.ipynb) at $\mu = 0$ (the kinematics
   that doomed the dwarfs, now holding starlight), and evaluate the CMB's pressure.
3. Derive $S = (4/3)aT^3V$ and the adiabat $TV^{1/3} = \text{const}$; then demonstrate
   numerically that stretching every mode by $s$ maps a 3000 K Planck spectrum *exactly* onto
   the Planck spectrum at $3000/s$ K.
4. Read the consequence (prose): a 3000 K spectrum from recombination arrives, 13.8 Gyr
   later, as a 2.725 K spectrum, cooled $\times$1100 and deformed not at all, which is precisely why the next exercise can exist.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    n_cmb,
    410.7,
    "411 photons per cubic centimetre, everywhere",
    rtol=1e-2,
)
validate.close(
    P_cmb,
    1.39e-14,
    "the sky's pressure: u/3 by the ultrarelativistic kinematics of §7.11, at μ = 0",
    rtol=2e-2,
)
validate.check(
    map_dev < 1e-12,
    "Planck maps onto Planck under expansion: stretched 3000 K lands on 2.725 K at machine level",
    f"max deviation {map_dev:.1e}",
)

## Exercise 6 — The jewel: fitting FIRAS

The most perfect blackbody ever measured, fit with one parameter. Cite {eq}`eq-firas-fit`.

1. Implement `B_nu_MJy(nu_Hz, T)` (conversions at their single stated points) and certify it
   against the canonical CMB peak (160.2 GHz, 383.9 MJy/sr) *before* any fitting; then
   certify the fit procedure itself on synthetic data of known temperature with FIRAS-scale
   noise.
2. Load the embedded FIRAS monopole table (Fixsen et al. 1996, Table 4; cm⁻¹ → Hz via the
   single conversion `CM1_TO_HZ`) and fit $T$ with `scipy.optimize.curve_fit` (`p0 = 3.0`,
   `sigma` from the table's uncertainties, `absolute_sigma=True`).
3. Report $T$ and its covariance-derived uncertainty, and plot data-on-curve with the
   residuals (in kJy/sr, with error bars) in an inset.
4. Close the century (prose): from a desperate interpolation in Berlin to a sky-filling
   spectrum matching it at parts in $10^4$; and the bookkeeping read out: photon-to-baryon
   $\sim1.6\times10^9$, the universe as mostly thermal light.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    B_peak,
    383.9,
    "the FIRAS-unit Planck function certified on the canonical peak before any fitting",
    rtol=1e-3,
)
validate.close(
    T_synth[0],
    T_CMB_REF,
    "the fit procedure certified: synthetic 50-ppm data returns its own temperature",
    atol=2e-4,
)
validate.close(
    T_cmb,
    2.725,
    "FIRAS: the most perfect blackbody, one free parameter",
    rtol=2e-3,
)
validate.close(
    photon_per_baryon,
    1.66e9,
    "cosmology's great dimensionless number: a billion and a half photons per baryon",
    rtol=5e-2,
)

## Exercise 7 — A planet in the balance

Two applications of $\sigma T^4$ and a 33-kelvin gap with a name. Cite {eq}`eq-earth-balance`.

1. Balance absorbed solar power $\pi R_E^2(1 - A)S_0$ against radiated $4\pi R_E^2\sigma
   T^4$ and derive $T_{\text{eq}} = T_\odot\sqrt{R_\odot/2d}\,(1 - A)^{1/4}$ (the planet's
   radius cancels).
2. Evaluate with `earth_equilibrium`: 278.3 K at $A = 0$ and 254.6 K at $A = 0.3$.
3. Compare with the observed mean 288 K and *name* the 33 K gap: the greenhouse effect as a
   frequency-selective blanket (one outward paragraph; climate physics cited, not developed).
4. Reflect (prose): the same law that weighed the Sun and fit the Big Bang's afterglow also
   sets the temperature of home; three scales, one integral from [§7.3](statistical-toolkit.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [T_black, T_eq],
    [278.3, 254.6],
    "Earth's radiative equilibrium, black and real",
    rtol=1e-2,
)
validate.check(
    32.0 < gap < 35.0,
    "and the named 33 K: the greenhouse gap between equilibrium and observation",
    f"gap = {gap:.1f} K",
)

## Exercise 8 — Where the story began, finished

The volume opened by promising to revisit the place where classical statistical mechanics
first broke in public, and this notebook kept that appointment with conspicuous economy. The
catastrophe was real (equipartition is not wrong, only unbounded) and the cure cost one
line, because [§7.5](oscillator-at-temperature.ipynb) had already warmed the oscillator, [§7.7](bose-einstein-fermi-dirac.ipynb) had already priced the occupation,
and [§7.3](statistical-toolkit.ipynb) had already counted the modes and done the integral. What the assembled law then
delivered: the Sun weighed to its displayed digits, a constant of nature that the SI now
defines rather than measures, a peak that splits in two under a change of variable, the
pressure that doomed white dwarfs now holding starlight, and — stretched a thousandfold
across the age of the universe — a spectrum whose fidelity to Planck's form is measured in
parts in ten thousand.

There is a fine irony in FIRAS. Planck introduced the quantum reluctantly, as a temporary
expedient he hoped one day to retire; a century later, the single most precisely Planckian
object ever observed is the universe itself. Nature not only kept the expedient — it built
the sky out of it.

The movement now turns to the exchange: how matter talks to this gas (Einstein's A and B
coefficients, and the laser they seeded, [§7.15](einstein-a-b-coefficients.ipynb)), how a solid hums the same law ([§7.16](phonons-debye.ipynb)), and
what happens when a boson gas whose number *is* conserved runs out of room ([§7.17](bose-einstein-condensation.ipynb)).

## Notebook summary

Movement IV's opener: the volume's origin story, told with the machinery to finish it.

- **The catastrophe** {eq}`eq-rayleigh-jeans`: Rayleigh–Jeans derived honestly from the equipartition of [§5.5](../05-classical-stat-mech/ergodicity.ipynb)
  on the mode count of [§7.3](statistical-toolkit.ipynb): the correct low-frequency limit (0.995 at $x = 0.01$,
  gated) whose integral grows as $\Lambda^3$ without bound ($\times8$ per doubling, gated).
- **$\mu = 0$ and the assembly** {eq}`eq-mu-zero`, {eq}`eq-planck-law`: unconserved number
  $\Rightarrow$ $\partial F/\partial N = 0$, the flag of [§7.7](bose-einstein-fermi-dirac.ipynb) argued in full (and what it kills:
  no cavity BEC); Planck assembled from [§7.5](oscillator-at-temperature.ipynb) + [§7.7](bose-einstein-fermi-dirac.ipynb) + [§7.3](statistical-toolkit.ipynb) in one line, both limits gated
  (0.995/0.951/0.582/0.034 against RJ; 1.0068 against Wien at $x = 5$).
- **Stefan–Boltzmann and the trap** {eq}`eq-stefan-boltzmann`, {eq}`eq-quad-trap`:
  dimensional `quad` returns $8\times10^{-38}$ against the true 0.84 J/m³ (thirty-seven orders, zero warnings, gated) while the nondimensionalized integral is exact to ten
  digits; $\sigma = 5.670374\times10^{-8}$ from constants (SI-exact); the Sun at
  $3.8280\times10^{26}$ W and 1361 W/m², gated.
- **Wien, twice** {eq}`eq-wien`: $x^* = 4.9651$ and $2.8214$ by `brentq`; $b = 2.8978$ mm·K
  on CODATA; the Sun's two peaks (502 vs 883 nm): a density's maximum lives in its variable.
- **Photon thermodynamics** {eq}`eq-photon-thermo`: 411 photons/cm³ (gated), $P = u/3$ by
  the kinematics of [§7.11](white-dwarfs-chandrasekhar.ipynb) ($1.4\times10^{-14}$ Pa for the sky), and Planck mapping onto Planck
  under expansion at machine level (gated): the CMB's survival, demonstrated.
- **FIRAS** {eq}`eq-firas-fit`: the embedded Fixsen et al. 1996 monopole table fit with one parameter, $T = 2.725$ K, after the unit function was certified on the canonical peak
  (383.9 MJy/sr at 160.2 GHz, itself predicted by Exercise 4) and the procedure on synthetic
  50-ppm data; residuals at parts in $10^4$; $1.7\times10^9$ photons per baryon.
- **The balance** {eq}`eq-earth-balance`: 254.6 K at $A = 0.3$ against the observed 288 K —
  the 33 K greenhouse gap, named and pointed outward.

Next door, matter and radiation negotiate: Einstein's A and B.

## Outlook

- **Einstein's A and B coefficients ([§7.15](einstein-a-b-coefficients.ipynb)).** Matter in equilibrium with this gas: detailed balance, stimulated emission, and the laser's seed.
- **Phonons ([§7.16](phonons-debye.ipynb)).** The same counting in a crystal: Debye's $T^3$, and the sound-speed
  blackbody inside every solid.
- **BEC ([§7.17](bose-einstein-condensation.ipynb)).** What $\mu = 0$ forbids here becomes forced there: condensation when the
  boson number *is* conserved and the excited states saturate.
- **Outward horizons, named.** The greenhouse problem in earnest (Pierrehumbert); stellar
  atmospheres and opacity; CMB anisotropies (the $10^{-5}$ ripples on this notebook's monopole) as the door to precision cosmology.
- **Cross-reference** [§7.3](statistical-toolkit.ipynb) (three invocations: modes, $\pi^4/15$, $\zeta(3)$), [§7.5](oscillator-at-temperature.ipynb) (the
  oscillator; `expm1`), [§7.7](bose-einstein-fermi-dirac.ipynb) ($\mu = 0$, argued here), [§7.11](white-dwarfs-chandrasekhar.ipynb) ($P = u/3$), [§5.5](../05-classical-stat-mech/ergodicity.ipynb) (equipartition,
  the honest villain), [§7.13](semiconductor-statistics.ipynb) (unit discipline, inherited).

In [ ]:
from ecp.style import footer

footer()